# Test Agent

Test de la classe `Agent` en utilisant un `MeteoDataset` comme base de connaissances.

In [1]:
# Uncomment if dependencies are not installed
# %pip install -r requirements.txt

In [2]:
import sys, os
sys.path.insert(0, os.path.abspath('..'))
sys.path.insert(0, os.path.abspath('.'))

from hybridiot import MeteoDataset, Agent
from datetime import datetime, timezone

## 1. Chargement du MeteoDataset

In [3]:
CSV_PATH = '../data/toulouse_meteo_live_jan2023(5stations).csv'

ds = MeteoDataset.from_csv(CSV_PATH)
print(f'Stations : {[s.id for s in ds.sensors]}')
print(f'Types    : {ds.data_types}')
print(f'Période  : {ds.start_time} -> {ds.end_time}')

  Sampling frequency detected: 15 min
Stations : ['Toulouse Avenue De Grande Bretagne', 'Toulouse Basso Cambo', 'Toulouse Montaudran', 'Toulouse Parc Jardin Des Plantes', 'Toulouse Soupetard']
Types    : ['temperature', 'humidite', 'pression', 'pluie', 'pluie_intensite_max']
Période  : 2023-01-01 00:00:00+00:00 -> 2023-01-31 23:45:00+00:00


## 2. Création d'un Agent

Un agent = une station + un type de donnée. Par exemple, l'agent qui mesure la température à Basso Cambo.

In [4]:
# Récupération de la station depuis le dataset pour obtenir position
station = ds.get_sensor('Toulouse Basso Cambo')
print(f'Station trouvée : {station}')

# Création de l'agent avec les mêmes métadonnées que la station
agent_temp = Agent(
    id         = station.id,
    position   = station.position,
    is_virtual = False,
    data_type  = 'temperature',
)
print(f'\nAgent créé : {agent_temp}')

Station trouvée : Sensor(id='Toulouse Basso Cambo', virtual=False, pos=(43.5700, 1.3900, 157.2m))

Agent créé : Agent(id='Toulouse Basso Cambo', position=Position(lon=1.39, lat=43.57, alt=157.2, X=0, Y=0)), is_virtual=False, data_type='temperature', frequency=15, dataset=[], fault_periods=[], environment=None)


## 3. Initialisation du dataset local

`load_from_dataset()` charge les valeurs de la base de connaissances **et** construit automatiquement `fault_periods` à partir de ces données.

In [5]:
agent_temp.load_from_dataset(ds)

print(f'Nombre de valeurs dans le dataset local : {len(agent_temp.dataset)}')
print(f'Nombre de fault_periods détectées      : {len(agent_temp.fault_periods)}')
print(f'\n5 premières valeurs :')
for ts, val in agent_temp.dataset[:5]:
    print(f'  {ts}  ->  {val}')
print(f'\n5 dernières valeurs :')
for ts, val in agent_temp.dataset[-5:]:
    print(f'  {ts}  ->  {val}')

Nombre de valeurs dans le dataset local : 2976
Nombre de fault_periods détectées      : 0

5 premières valeurs :
  2023-01-01 00:00:00+00:00  ->  13.199999809265137
  2023-01-01 00:15:00+00:00  ->  13.300000190734863
  2023-01-01 00:30:00+00:00  ->  13.399999618530273
  2023-01-01 00:45:00+00:00  ->  13.5
  2023-01-01 01:00:00+00:00  ->  13.600000381469727

5 dernières valeurs :
  2023-01-31 22:45:00+00:00  ->  7.699999809265137
  2023-01-31 23:00:00+00:00  ->  7.599999904632568
  2023-01-31 23:15:00+00:00  ->  7.5
  2023-01-31 23:30:00+00:00  ->  7.400000095367432
  2023-01-31 23:45:00+00:00  ->  7.400000095367432


In [6]:
# Affichage des fault_periods détectées dans la base de connaissances
if agent_temp.fault_periods:
    print(f'Périodes anormales détectées dans le dataset local :')
    for i, (start, end) in enumerate(agent_temp.fault_periods):
        duration = (end - start).total_seconds() / 60
        print(f'  [{i}]  {start}  ->  {end}   (durée : {duration:.0f} min)')
else:
    print('Aucune période anormale détectée.')

Aucune période anormale détectée.


## 4. Simulation d'un flux de données avec add_value()

`add_value()` ajoute une nouvelle valeur et met à jour `fault_periods` automatiquement.

In [7]:
n_before          = len(agent_temp.dataset)
n_faults_before   = len(agent_temp.fault_periods)
print(f'Avant  : {n_before} valeurs, {n_faults_before} fault_periods')

# Ajout d'une valeur normale avec timestamp courant (UTC)
agent_temp.add_value(value=22.5)
print(f'\nAprès add_value(22.5) :')
print(f'  dernière valeur : {agent_temp.dataset[-1]}')
print(f'  fault_periods   : {len(agent_temp.fault_periods)} (inchangé car valeur normale)')

# Ajout d'une valeur avec timestamp spécifique
ts_custom = datetime(2023, 2, 1, 12, 0, tzinfo=timezone.utc)
agent_temp.add_value(value=18.3, timestamp=ts_custom)
print(f'\nAprès add_value(18.3, ts spécifique) :')
print(f'  dernière valeur : {agent_temp.dataset[-1]}')

print(f'\nAprès : {len(agent_temp.dataset)} valeurs')

Avant  : 2976 valeurs, 0 fault_periods

Après add_value(22.5) :
  dernière valeur : (datetime.datetime(2026, 7, 8, 13, 46, 38, 9881, tzinfo=datetime.timezone.utc), 22.5)
  fault_periods   : 0 (inchangé car valeur normale)

Après add_value(18.3, ts spécifique) :
  dernière valeur : (datetime.datetime(2023, 2, 1, 12, 0, tzinfo=datetime.timezone.utc), 18.3)

Après : 2978 valeurs


## 5. Vérification avec is_fault()

In [8]:
# Sans timestamp -> vérifie la dernière valeur
print(f'is_fault() (dernière valeur) : {agent_temp.is_fault()}')
print(f'Dernière valeur : {agent_temp.dataset[-1]}')

# Avec timestamp spécifique
ts_check = agent_temp.dataset[0][0]
print(f'\nis_fault({ts_check}) : {agent_temp.is_fault(timestamp=ts_check)}')
print(f'Valeur correspondante : {agent_temp.dataset[0][1]}')

is_fault() (dernière valeur) : False
Dernière valeur : (datetime.datetime(2023, 2, 1, 12, 0, tzinfo=datetime.timezone.utc), 18.3)

is_fault(2023-01-01 00:00:00+00:00) : False
Valeur correspondante : 13.199999809265137


## 6. Test de la détection automatique des fault_periods

Vérification que `add_value()` met bien à jour `fault_periods` quand on injecte des valeurs anormales successives.

In [9]:
n_faults_before = len(agent_temp.fault_periods)
print(f'Avant : {n_faults_before} fault_periods')

# Valeur anormale (-100°C, hors range)
ts1 = datetime(2023, 2, 2, 0, 0, tzinfo=timezone.utc)
agent_temp.add_value(value=-100.0, timestamp=ts1)
print(f'\nAjout valeur -100 à {ts1}')
print(f'  fault_periods : {len(agent_temp.fault_periods)} = nouvelle période ouverte')
print(f'  dernière      : {agent_temp.fault_periods[-1]}')

# Deuxième valeur anormale consécutive (NaN)
import math
ts2 = datetime(2023, 2, 2, 0, 15, tzinfo=timezone.utc)
agent_temp.add_value(value=math.nan, timestamp=ts2)
print(f'\nAjout valeur NaN à {ts2}')
print(f'  fault_periods : {len(agent_temp.fault_periods)} = période étendue (pas nouvelle)')
print(f'  dernière      : {agent_temp.fault_periods[-1]}')

# Troisième valeur anormale consécutive
ts3 = datetime(2023, 2, 2, 0, 30, tzinfo=timezone.utc)
agent_temp.add_value(value=200.0, timestamp=ts3)
print(f'\nAjout valeur 200 à {ts3}')
print(f'  fault_periods : {len(agent_temp.fault_periods)} = période étendue (pas nouvelle)')
print(f'  dernière      : {agent_temp.fault_periods[-1]}')

# Valeur normale -> ferme la période
ts4 = datetime(2023, 2, 2, 0, 45, tzinfo=timezone.utc)
agent_temp.add_value(value=15.0, timestamp=ts4)
print(f'\nAjout valeur 15.0 (normale) à {ts4}')
print(f'  fault_periods : {len(agent_temp.fault_periods)}  = inchangé, période figée')
print(f'  dernière      : {agent_temp.fault_periods[-1]}')

# Nouvelle valeur anormale -> nouvelle période
ts5 = datetime(2023, 2, 2, 1, 0, tzinfo=timezone.utc)
agent_temp.add_value(value=-50.0, timestamp=ts5)
print(f'\nAjout valeur -50 à {ts5}')
print(f'  fault_periods : {len(agent_temp.fault_periods)}  = nouvelle période ouverte')
print(f'  dernière      : {agent_temp.fault_periods[-1]}')

Avant : 0 fault_periods

Ajout valeur -100 à 2023-02-02 00:00:00+00:00
  fault_periods : 1 = nouvelle période ouverte
  dernière      : (datetime.datetime(2023, 2, 2, 0, 0, tzinfo=datetime.timezone.utc), datetime.datetime(2023, 2, 2, 0, 0, tzinfo=datetime.timezone.utc))

Ajout valeur NaN à 2023-02-02 00:15:00+00:00
  fault_periods : 1 = période étendue (pas nouvelle)
  dernière      : (datetime.datetime(2023, 2, 2, 0, 0, tzinfo=datetime.timezone.utc), datetime.datetime(2023, 2, 2, 0, 15, tzinfo=datetime.timezone.utc))

Ajout valeur 200 à 2023-02-02 00:30:00+00:00
  fault_periods : 1 = période étendue (pas nouvelle)
  dernière      : (datetime.datetime(2023, 2, 2, 0, 0, tzinfo=datetime.timezone.utc), datetime.datetime(2023, 2, 2, 0, 30, tzinfo=datetime.timezone.utc))

Ajout valeur 15.0 (normale) à 2023-02-02 00:45:00+00:00
  fault_periods : 1  = inchangé, période figée
  dernière      : (datetime.datetime(2023, 2, 2, 0, 0, tzinfo=datetime.timezone.utc), datetime.datetime(2023, 2, 2, 0, 

## 7. slice() — fenêtre comme liste de tuples

In [10]:
# Fenêtre sans filtres -> tout le dataset
all_data = agent_temp.slice()
print(f'Sans filtres : {len(all_data)} entries')

# Fenêtre sur une semaine
start = datetime(2023, 1, 1,  0,  0, tzinfo=timezone.utc)
end   = datetime(2023, 1, 7, 23, 45, tzinfo=timezone.utc)
week  = agent_temp.slice(start_time=start, end_time=end)
print(f'Première semaine de janvier 2023 : {len(week)} entries')
print(f'\n3 premières entries de la fenêtre :')
for ts, val in week[:3]:
    print(f'  {ts}  ->  {val}')

Sans filtres : 2983 entries
Première semaine de janvier 2023 : 672 entries

3 premières entries de la fenêtre :
  2023-01-01 00:00:00+00:00  ->  13.199999809265137
  2023-01-01 00:15:00+00:00  ->  13.300000190734863
  2023-01-01 00:30:00+00:00  ->  13.399999618530273
